In [1]:
import json

with open("C:/Users/smart/Downloads/APIBench_Q/APIBench_Q/Python/ReformulatedPythonQueries.json", "r") as file:
    data = json.load(file)

In [2]:
import json
import pandas as pd

df = pd.DataFrame(data)
print(df.head())

                                                                        1  \
APIs                                                 [random.randrange()]   
APIClasses                                                       [random]   
OriginalQuery            How to generate a "big" random number in Python?   
Source                                                     Stack Overflow   
@NLPAUG_Delete_Random@  [How to " big " random number in?, How to gene...   

                                                                        2  \
APIs                                                    [sys.getsizeof()]   
APIClasses                                                          [sys]   
OriginalQuery             Storing big numbers over 9,000 digits in Python   
Source                                                     Stack Overflow   
@NLPAUG_Delete_Random@  [Storing numbers 9, digits in Python, Storing ...   

                                                                        3 

In [3]:
import json
import pandas as pd

# Convert to list of dictionaries (flattening the nested structure)
records = []
for key, entry in data.items():
    query = entry.get("OriginalQuery", "")
    apis = entry.get("APIs", [])
    api_classes = entry.get("APIClasses", [])
    
    # Optional: Combine APIs and Classes into a single API call string
    api_call = ", ".join(apis)
    
    # Simulate API documentation or meta information (can be expanded later)
    api_doc = "API Classes: " + ", ".join(api_classes) if api_classes else "N/A"
    
    records.append({
        "instruction": query,
        "api_call": api_call,
        "api_doc": api_doc
    })

# Create a DataFrame
df = pd.DataFrame(records)

# Drop missing or incomplete rows
df.dropna(subset=["instruction", "api_call"], inplace=True)

# Optional: View structured data
print(df.head(3))

# Format for training
def format_prompt(row):
    return {
        "prompt": f"{row['instruction']}\nUse this API documentation for reference:\n{row['api_doc']}",
        "response": row['api_call']
    }

formatted_data = df.apply(format_prompt, axis=1).tolist()

# Preview one
print(formatted_data[0])


                                         instruction            api_call  \
0   How to generate a "big" random number in Python?  random.randrange()   
1    Storing big numbers over 9,000 digits in Python     sys.getsizeof()   
2  Python: Calculate factorial of a non-integral ...        math.gamma()   

               api_doc  
0  API Classes: random  
1     API Classes: sys  
2    API Classes: math  
{'prompt': 'How to generate a "big" random number in Python?\nUse this API documentation for reference:\nAPI Classes: random', 'response': 'random.randrange()'}


In [4]:
import json

# Save the formatted dataset
with open("C:/Users/smart/Downloads/formatted_apidata.json", "w") as f:
    for item in formatted_data:
        f.write(json.dumps(item) + "\n")

In [7]:
import json

# Input file
input_path = "C:/Users/smart/Downloads/formatted_apidata.json"

# Output files
output_with_retriever = "C:/Users/smart/Downloads/chat_data_with_retriever.jsonl"
output_without_retriever = "C:/Users/smart/Downloads/chat_data_without_retriever.jsonl"

with open(input_path, "r") as infile, \
     open(output_with_retriever, "w") as out_with, \
     open(output_without_retriever, "w") as out_without:

    for line in infile:
        item = json.loads(line)
        prompt = item["prompt"]
        response = item["response"]

        # Split instruction and API doc
        if "Use this API documentation for reference:" in prompt:
            parts = prompt.split("Use this API documentation for reference:")
            instruction = parts[0].strip()
            api_doc = parts[1].strip()
        else:
            instruction = prompt.strip()
            api_doc = ""

        # Chat format WITH retriever (instruction + api_doc)
        chat_with = {
            "conversations": [
                {"role": "user", "content": f"{instruction}\nUse this API documentation for reference:\n{api_doc}"},
                {"role": "assistant", "content": response}
            ]
        }

        # Chat format WITHOUT retriever (only instruction)
        chat_without = {
            "conversations": [
                {"role": "user", "content": instruction},
                {"role": "assistant", "content": response}
            ]
        }

        # Write to respective files
        out_with.write(json.dumps(chat_with) + "\n")
        out_without.write(json.dumps(chat_without) + "\n")

print("Chat-style data generated successfully.")

Chat-style data generated successfully.
